# ISL GafferAI — Fotmob Data Exploration
**Goal:** Understand what data Fotmob has for ISL and whether it is sufficient to build GafferAI for Indian Super League

Pipeline we are aiming for:
```
Fotmob API → Player Match History → Feature Engineering → LightGBM → ILP Squad Optimizer
```

## Cell 0 — Install & Imports

In [7]:


import asyncio
import pandas as pd
import json
# First let's see what is actually in the library
from fotmob.fotmob import FotMob


print('All imports successful')

All imports successful


## Cell 1 — Connect to Fotmob & Get ISL League Info

In [12]:
from fotmob.fotmob import FotMob

client = FotMob()

# See all available methods
methods = [m for m in dir(client) if not m.startswith('_')]
print(methods)

['checkDate', 'getLeague', 'getMatchDetails', 'getMatchesByDate', 'getPlayer', 'getTeam']


In [15]:
import inspect
print(inspect.signature(client.getLeague))
print(inspect.getsource(client.getLeague))

(tab='overview', type='league', timeZone='America/New_York')
    def getLeague(id, tab = "overview", type = "league", timeZone = "America/New_York"):
        url = leaguesUrl + f"id={id}&tab={tab}&type={type}&timeZone={timeZone}"
        print(url)

        with urllib.request.urlopen(url) as _url:
            data = json.loads(_url.read().decode())
            return data



In [17]:
# Call on class directly, not instance
league = FotMob.getLeague(7524)

print('Type:', type(league))
print('Keys:', league.keys())

https://www.fotmob.com/leagues?id=7524&tab=overview&type=league&timeZone=America/New_York


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Cell 2 — Explore League Structure

In [ ]:
# See what raw data comes back
# This tells us what is available at league level

try:
    print(json.dumps(league.__dict__, indent=2, default=str)[:3000])
except:
    print(league)

## Cell 3 — Get ISL Matches (Fixtures)

In [ ]:
# Get all ISL matches for current season
# We need match IDs to then get player stats per match

try:
    matches_data = await client.get_league(
        id=7524,
        tab='matches',
        type='league',
        season='2024/2025'
    )
    print(type(matches_data))
    print(dir(matches_data))
except Exception as e:
    print(f'Error: {e}')

## Cell 4 — Get a Single Match Details

In [ ]:
# Try fetching a known recent ISL match
# We will find a match ID from the league data above
# For now try a sample match ID — adjust based on Cell 3 output

# Try to extract match IDs from league data first
try:
    # Common structure in fotmob-api
    if hasattr(matches_data, 'matches'):
        all_matches = matches_data.matches
        print(f'Total matches found: {len(all_matches)}')
        print('\nFirst match sample:')
        print(all_matches[0])
    else:
        print('Matches attribute not found, checking raw structure...')
        print(dir(matches_data))
except Exception as e:
    print(f'Error: {e}')

## Cell 5 — Get Player Stats from a Match

In [ ]:
# Once we have a match ID, get full match details
# This is where xG, minutes, ratings, goals, assists all live

# Replace MATCH_ID with an actual ID from Cell 4
SAMPLE_MATCH_ID = None  # We will fill this after Cell 4 runs

if SAMPLE_MATCH_ID:
    try:
        match = await client.get_match_details(match_id=SAMPLE_MATCH_ID)
        print('Match details keys:', dir(match))
        
        # Try to get player stats
        if hasattr(match, 'content'):
            print('\nContent keys:', dir(match.content))
        if hasattr(match, 'players'):
            print('\nPlayers found!')
            print(match.players)
    except Exception as e:
        print(f'Error: {e}')
else:
    print('Fill SAMPLE_MATCH_ID from Cell 4 output first')

## Cell 6 — Get League Player Stats

In [ ]:
# Fotmob has a league stats endpoint that aggregates player stats
# This might give us season totals per player directly

try:
    stats = await client.get_league(
        id=7524,
        tab='stats',
        type='league',
        season='2024/2025'
    )
    print(type(stats))
    print(dir(stats))
    
    # Try to see player stats
    try:
        print(json.dumps(stats.__dict__, indent=2, default=str)[:3000])
    except:
        print(stats)
        
except Exception as e:
    print(f'Error: {e}')

## Cell 7 — Try Getting a Specific Player Stats

In [ ]:
# If we can get player IDs, we can fetch individual player career stats
# Let's try with a known ISL player
# Sandesh Jhingan is a well known ISL player — find his Fotmob ID

try:
    # Search for a player
    search_result = await client.get_search_results(query='Sandesh Jhingan')
    print(type(search_result))
    print(search_result)
except Exception as e:
    print(f'Search error: {e}')
    print('Try different approach...')

## Cell 8 — Summary: What Data Is Available?

After running all cells above, fill in what we found:

| Data Type | Available? | Notes |
|-----------|-----------|-------|
| Match list | ? | |
| Player IDs | ? | |
| Minutes played | ? | |
| Goals / Assists | ? | |
| xG / xA | ? | |
| FotMob rating | ? | |
| Match by match history | ? | |

**Decision after exploration:**
- If match-by-match player stats available → Full GafferAI pipeline is buildable
- If only season totals → Limited but can still build a simpler version
- If data is too sparse → May need alternative data source